# Run J: D-FINE-S, 640 px, one-class small-defect detection

D-FINE was published at ICLR 2025. This notebook uses the **same prepared data, label matching, seed-42 70/15/15 split, validation set, and overall/small/medium/large test sets** as `yolov8s-640-2.ipynb`.

It fixes the earlier NaN-box failure by using the learning rate scaled to batch 8 and disabling AMP. Enable Kaggle Internet and a T4 GPU, attach **SmallDefectPreprocessing**, and run as a Save Version.


In [ ]:
import ast
import csv
import json
import os
import random
import re
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml
from PIL import Image

assert Path("/kaggle/input").exists(), "This notebook must run on Kaggle."


In [ ]:
# Run identity and fixed comparison settings.
RUN_NAME = "RunJ_dfine_s_imgsz640"
MODEL_LABEL = "D-FINE-S"
WANDB_PROJECT = "smallDefectDetection"

DATASET_NAMES = [
    "DAGM",
    "GC10-DET",
    "KolektorSDD2",
    "MPDD",
    "MTD",
    "Severstal",
    "VisA",
]
SIZE_BUCKETS = ["small", "medium", "large"]

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 50
WORKERS = 2
DEVICE = "cuda"

# D-FINE-S's official custom config is tuned for global batch 64 and lr 4e-4.
# This is the linear batch-8 equivalent: 4e-4 * 8 / 64 = 5e-5.
BASE_LR = 0.00005
BACKBONE_LR = 0.000025

BASE_DIR = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREPARED_DATASET_DIR = BASE_DIR / "run_a_yolo_dataset"
DFINE_REPO = BASE_DIR / "D-FINE"
COCO_ROOT = BASE_DIR / "dfine_defect_coco"
RUN_DIR = BASE_DIR / "dfine_runs" / RUN_NAME
FINAL_OUTPUT_DIR = BASE_DIR / "final_outputs" / RUN_NAME


In [ ]:
# Kaggle Internet must be enabled for this cell.
if not DFINE_REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Peterande/D-FINE.git", str(DFINE_REPO)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(DFINE_REPO / "requirements.txt"), "pycocotools"],
    check=True,
)
print("D-FINE ready at:", DFINE_REPO)


In [ ]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

expected_datasets = set(DATASET_NAMES)

print("Searching for dataset root under:", KAGGLE_INPUT_ROOT)

valid_roots = []

for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
    root_path = Path(root)
    dir_set = set(dirs)

    matches = expected_datasets.intersection(dir_set)

    if len(matches) >= 5:
        valid_roots.append(root_path)
        print("Found candidate:", root_path)
        print("Matches:", sorted(matches))

if not valid_roots:
    raise FileNotFoundError(
        "Could not find a folder containing the expected dataset folders: "
        f"{sorted(expected_datasets)}"
    )

hf_dataset_path = valid_roots[0]

print("\nUsing dataset root:", hf_dataset_path)
print("Available datasets:", sorted([p.name for p in hf_dataset_path.iterdir() if p.is_dir()]))

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

samples = []
missing_count = 0

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        image_dir = hf_dataset_path / dataset_name / size_bucket / "images"
        label_dir = hf_dataset_path / dataset_name / size_bucket / "labels_yolo"

        if not image_dir.exists() or not label_dir.exists():
            print("Missing directory:", dataset_name, size_bucket)
            continue

        # Index labels once instead of checking the filesystem per image
        label_index = {
            label_path.stem: label_path
            for label_path in label_dir.glob("*.txt")
        }

        matched = 0
        bucket_missing = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            image_stem = image_path.stem
            base_stem = image_stem.removesuffix("_defect")

            possible_label_stems = [
                image_stem,
                image_stem.replace("_defect", "_bbs"),
                f"{base_stem}_bbs",
            ]

            label_path = next(
                (
                    label_index[stem]
                    for stem in possible_label_stems
                    if stem in label_index
                ),
                None,
            )

            if label_path is None:
                bucket_missing += 1
                continue

            samples.append({
                "image_path": image_path,
                "label_path": label_path,
                "dataset": dataset_name,
                "size": size_bucket,
                "stratum": f"{dataset_name}_{size_bucket}",
            })

            matched += 1

        missing_count += bucket_missing
        print(
            f"{dataset_name}/{size_bucket}: "
            f"{matched} matched, {bucket_missing} missing"
        )

print("\nTotal usable samples:", len(samples))
print("Total missing labels:", missing_count)

if not samples:
    raise RuntimeError("No image-label pairs were matched.")

In [ ]:
import random
from collections import defaultdict

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

random.seed(SEED)

by_stratum = defaultdict(list)

for sample in samples:
    by_stratum[sample["stratum"]].append(sample)

train_samples = []
val_samples = []
test_samples = []

for stratum, group in sorted(by_stratum.items()):
    random.shuffle(group)

    n = len(group)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    train_samples.extend(group[:n_train])
    val_samples.extend(group[n_train:n_train + n_val])
    test_samples.extend(group[n_train + n_val:])

random.shuffle(train_samples)
random.shuffle(val_samples)
random.shuffle(test_samples)

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))
print("Total:", len(train_samples) + len(val_samples) + len(test_samples))

assert len(train_samples) > 0
assert len(val_samples) > 0
assert len(test_samples) > 0

In [ ]:
from collections import Counter

def count_by_size(samples, split_name):
    counts = Counter(s["size"] for s in samples)
    total = len(samples)

    print()
    print(split_name)
    print("Total :", total)
    print("Small :", counts["small"])
    print("Medium:", counts["medium"])
    print("Large :", counts["large"])

count_by_size(train_samples, "Train")
count_by_size(val_samples, "Val")
count_by_size(test_samples, "Test")

In [ ]:
def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


reset_dir(PREPARED_DATASET_DIR)

splits_to_make = [
    "train",
    "val",
    "test",
    "test_small",
    "test_medium",
    "test_large",
]

for split in splits_to_make:
    (PREPARED_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (PREPARED_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Prepared dataset dir:", PREPARED_DATASET_DIR)

In [ ]:
def rewrite_label_as_single_class(src_label_path, dst_label_path):
    new_lines = []

    with open(src_label_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) < 5:
                continue

            coords = parts[1:5]
            new_lines.append("0 " + " ".join(coords))

    with open(dst_label_path, "w") as f:
        f.write("\n".join(new_lines))


def export_split(split_name, split_samples):
    for idx, sample in enumerate(split_samples):
        src_image = sample["image_path"]
        src_label = sample["label_path"]

        safe_name = f"{sample['dataset']}_{sample['size']}_{idx}_{src_image.name}"

        dst_image = PREPARED_DATASET_DIR / "images" / split_name / safe_name
        dst_label = PREPARED_DATASET_DIR / "labels" / split_name / f"{Path(safe_name).stem}.txt"

        shutil.copy2(src_image, dst_image)
        rewrite_label_as_single_class(src_label, dst_label)


export_split("train", train_samples)
export_split("val", val_samples)
export_split("test", test_samples)

test_small_samples = [s for s in test_samples if s["size"] == "small"]
test_medium_samples = [s for s in test_samples if s["size"] == "medium"]
test_large_samples = [s for s in test_samples if s["size"] == "large"]

export_split("test_small", test_small_samples)
export_split("test_medium", test_medium_samples)
export_split("test_large", test_large_samples)

print("YOLO dataset prepared.")
print("Test overall:", len(test_samples))
print("Test small:", len(test_small_samples))
print("Test medium:", len(test_medium_samples))
print("Test large:", len(test_large_samples))

In [ ]:
def write_data_yaml(path, test_split):
    data_yaml = {
        "path": str(PREPARED_DATASET_DIR),
        "train": "images/train",
        "val": "images/val",
        "test": f"images/{test_split}",
        "nc": 1,
        "names": ["defect"],
    }

    with open(path, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

data_yaml_path = PREPARED_DATASET_DIR / "data.yaml"

data_yaml_all = Path("/kaggle/working/data_all.yaml")
data_yaml_small = Path("/kaggle/working/data_small.yaml")
data_yaml_medium = Path("/kaggle/working/data_medium.yaml")
data_yaml_large = Path("/kaggle/working/data_large.yaml")

write_data_yaml(data_yaml_path, "test")
write_data_yaml(data_yaml_all, "test")
write_data_yaml(data_yaml_small, "test_small")
write_data_yaml(data_yaml_medium, "test_medium")
write_data_yaml(data_yaml_large, "test_large")

print(data_yaml_path)
print(data_yaml_path.read_text())

In [ ]:
for split in ["train", "val", "test", "test_small", "test_medium", "test_large"]:
    image_count = len(list((PREPARED_DATASET_DIR / "images" / split).glob("*")))
    label_count = len(list((PREPARED_DATASET_DIR / "labels" / split).glob("*.txt")))

    print(split)
    print(" images:", image_count)
    print(" labels:", label_count)


In [ ]:
# Convert the exact fixed YOLO dataset to COCO detection JSON for D-FINE.
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLITS = ("train", "val", "test", "test_small", "test_medium", "test_large")

def yolo_box_to_coco(parts, image_width, image_height):
    if len(parts) < 5:
        return None
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = max(0.0, min((cx - bw / 2) * image_width, image_width))
    y1 = max(0.0, min((cy - bh / 2) * image_height, image_height))
    x2 = max(0.0, min((cx + bw / 2) * image_width, image_width))
    y2 = max(0.0, min((cy + bh / 2) * image_height, image_height))
    width, height = x2 - x1, y2 - y1
    if width <= 0 or height <= 0:
        return None
    return [round(x1, 4), round(y1, 4), round(width, 4), round(height, 4)]

def convert_split_to_coco(split_name):
    image_dir = PREPARED_DATASET_DIR / "images" / split_name
    label_dir = PREPARED_DATASET_DIR / "labels" / split_name
    images, annotations = [], []
    annotation_id = 1
    skipped_boxes = 0

    for image_id, image_path in enumerate(sorted(image_dir.iterdir()), start=1):
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        with Image.open(image_path) as image:
            width, height = image.size
        images.append({"id": image_id, "file_name": image_path.name, "width": width, "height": height})

        label_path = label_dir / f"{image_path.stem}.txt"
        for line in label_path.read_text().splitlines():
            box = yolo_box_to_coco(line.split(), width, height)
            if box is None:
                skipped_boxes += 1
                continue
            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 0,
                "bbox": box,
                "area": round(box[2] * box[3], 4),
                "iscrowd": 0,
            })
            annotation_id += 1

    coco = {
        "info": {"description": "One-class small-defect D-FINE dataset"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [{"id": 0, "name": "defect", "supercategory": "defect"}],
    }
    annotation_dir = COCO_ROOT / "annotations"
    annotation_dir.mkdir(parents=True, exist_ok=True)
    output_path = annotation_dir / f"instances_{split_name}.json"
    output_path.write_text(json.dumps(coco, indent=2))
    return {"split": split_name, "images": len(images), "instances": len(annotations),
            "skipped_boxes": skipped_boxes, "annotation_file": str(output_path)}

conversion_rows = [convert_split_to_coco(split) for split in SPLITS]
conversion_df = pd.DataFrame(conversion_rows)
display(conversion_df)

expected_images = {"train": 8858, "val": 1892, "test": 1920, "test_small": 462, "test_medium": 868, "test_large": 590}
for row in conversion_rows:
    assert row["images"] == expected_images[row["split"]], row
    assert row["instances"] > 0 and row["skipped_boxes"] == 0, row


In [ ]:
# Write the D-FINE config. The previous failure was NaN box predictions.
# Two stability fixes are deliberate: batch-scaled LR and full precision (no AMP).
CONFIG_DIR = DFINE_REPO / "configs" / "dfine" / "custom"
TRAIN_CONFIG = CONFIG_DIR / "dfine_defect_s_stable.yml"

def write_config(path, image_split, annotation_split, output_dir):
    image_dir = PREPARED_DATASET_DIR / "images" / image_split
    annotation_file = COCO_ROOT / "annotations" / f"instances_{annotation_split}.json"
    path.write_text(f'''__include__: [ './dfine_hgnetv2_s_custom.yml' ]

output_dir: {output_dir}
num_classes: 1
remap_mscoco_category: False
epochs: {EPOCHS}

# Do not enable AMP. The earlier AMP run produced NaN predicted boxes.
use_amp: False
scaler:
  enabled: False

optimizer:
  type: AdamW
  lr: {BASE_LR}
  betas: [0.9, 0.999]
  weight_decay: 0.0001
  params:
    - params: '^(?=.*backbone)(?!.*norm|bn).*$'
      lr: {BACKBONE_LR}
    - params: '^(?=.*backbone)(?=.*norm|bn).*$'
      lr: {BACKBONE_LR}
      weight_decay: 0.0
    - params: '^(?=.*(?:encoder|decoder))(?=.*(?:norm|bn|bias)).*$'
      weight_decay: 0.0

train_dataloader:
  total_batch_size: {BATCH_SIZE}
  num_workers: {WORKERS}
  dataset:
    img_folder: {PREPARED_DATASET_DIR / 'images' / 'train'}
    ann_file: {COCO_ROOT / 'annotations' / 'instances_train.json'}

val_dataloader:
  total_batch_size: {BATCH_SIZE}
  num_workers: {WORKERS}
  dataset:
    img_folder: {image_dir}
    ann_file: {annotation_file}
''')

write_config(TRAIN_CONFIG, "val", "val", RUN_DIR)
print(TRAIN_CONFIG.read_text())


In [ ]:
# Full-precision D-FINE-S training. Do not add --use-amp.
command = [sys.executable, "train.py", "-c", str(TRAIN_CONFIG), "--seed", str(SEED)]
print("Running:", " ".join(command))
subprocess.run(command, cwd=DFINE_REPO, check=True)


In [ ]:
# Locate the best checkpoint saved by D-FINE.
checkpoints = sorted(RUN_DIR.rglob("*.pth"), key=lambda item: item.stat().st_mtime, reverse=True)
if not checkpoints:
    raise FileNotFoundError(f"No D-FINE checkpoint found under {RUN_DIR}")
best_named = [item for item in checkpoints if "best" in item.name.lower()]
BEST_CHECKPOINT = best_named[0] if best_named else checkpoints[0]
print("Using checkpoint:", BEST_CHECKPOINT)


In [ ]:
# Validate and test the same best checkpoint on overall/small/medium/large splits.
evaluation_sets = {
    "val": ("val", "val"),
    "overall": ("test", "test"),
    "small": ("test_small", "test_small"),
    "medium": ("test_medium", "test_medium"),
    "large": ("test_large", "test_large"),
}

EVAL_LOG_DIR = FINAL_OUTPUT_DIR / "evaluation_logs"
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
evaluation_logs = {}
for name, (image_split, annotation_split) in evaluation_sets.items():
    eval_config = CONFIG_DIR / f"dfine_defect_s_{name}.yml"
    eval_output_dir = BASE_DIR / "dfine_evaluations" / RUN_NAME / name
    write_config(eval_config, image_split, annotation_split, eval_output_dir)
    command = [sys.executable, "train.py", "-c", str(eval_config), "--test-only", "-r", str(BEST_CHECKPOINT)]
    print("Running evaluation:", name)
    completed = subprocess.run(command, cwd=DFINE_REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = EVAL_LOG_DIR / f"{name}.log"
    log_path.write_text(completed.stdout)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"{name} evaluation failed. Read {log_path}")
    evaluation_logs[name] = log_path


In [ ]:
# Parse D-FINE's own validator and COCO metrics into an Excel-style summary.
def parse_evaluation(log_path):
    text = Path(log_path).read_text()
    validator_match = re.findall(r"Metrics:\s*(\{.*?\})", text)
    validator = ast.literal_eval(validator_match[-1]) if validator_match else {}

    ap50_95 = re.findall(r"Average Precision.*?IoU=0\.50:0\.95.*?area=\s*all.*?=\s*([0-9.]+)", text, flags=re.S)
    ap50 = re.findall(r"Average Precision.*?IoU=0\.50\s*\|.*?area=\s*all.*?=\s*([0-9.]+)", text, flags=re.S)
    return {
        "mAP50": float(ap50[-1]) if ap50 else float("nan"),
        "mAP50_95": float(ap50_95[-1]) if ap50_95 else float("nan"),
        "Precision": validator.get("precision", float("nan")),
        "Recall": validator.get("recall", float("nan")),
        "FPs": validator.get("FPs", float("nan")),
    }

metrics = {name: parse_evaluation(path) for name, path in evaluation_logs.items()}
overall, small, medium, large = (metrics[key] for key in ("overall", "small", "medium", "large"))
summary_row = {
    "Experiment": RUN_NAME,
    "Model": MODEL_LABEL,
    "Batch": BATCH_SIZE,
    "Epochs": EPOCHS,
    "mAP50": overall["mAP50"],
    "mAP50_95": overall["mAP50_95"],
    "Precision": overall["Precision"],
    "Recall": overall["Recall"],
    "mAP50_Small": small["mAP50"],
    "mAP50_Medium": medium["mAP50"],
    "mAP50_Large": large["mAP50"],
    "Recall_Small": small["Recall"],
    "Recall_Medium": medium["Recall"],
    "Recall_Large": large["Recall"],
    "Inference_Time_ms": float("nan"),
    "FP_Per_Image": overall["FPs"] / 1920 if pd.notna(overall["FPs"]) else float("nan"),
    "Notes": "D-FINE-S full precision, batch-scaled LR, fixed 640 split",
}

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(FINAL_OUTPUT_DIR / "summary.csv", index=False)
pd.DataFrame([{ "split": split, **values } for split, values in metrics.items()]).to_csv(FINAL_OUTPUT_DIR / "evaluation_metrics.csv", index=False)
pd.DataFrame(conversion_rows).to_csv(FINAL_OUTPUT_DIR / "split_counts.csv", index=False)
shutil.copy2(TRAIN_CONFIG, FINAL_OUTPUT_DIR / TRAIN_CONFIG.name)
shutil.copy2(BEST_CHECKPOINT, FINAL_OUTPUT_DIR / "best_checkpoint.pth")
shutil.copytree(COCO_ROOT / "annotations", FINAL_OUTPUT_DIR / "coco_annotations", dirs_exist_ok=True)

print("Saved final artifacts to:", FINAL_OUTPUT_DIR)
display(summary_df)
